# Week 4 Lab. Poisson Processes, Counts and Gaps
**Time Series Analysis & Random Processes** · Graduate School of Data Science, Chonnam National University · notebook v1 (2026-09-22)

---

### What this lab does

Last week the clock ticked. A state moved from day to day and we asked where it was after `n` steps.
This week nothing ticks. Events land at any instant, and there are two ways to look at the same timeline.

**Counts.** How many events have happened by time `t`?
**Gaps.** How long until the next one?

The lab's job is to make those two readings meet. One simulator generates the gaps; every diagnostic afterwards
reads the counts. When the numbers agree, the bridge on the deck is no longer a claim on a slide.

> 요약: 지난주는 틱 위에서 움직였고 이번 주는 아무 순간에나 사건이 생깁니다. 같은 타임라인을 「몇 건 일어났나」와 「다음까지 얼마나」 두 가지로 읽습니다. 간격으로 만든 경로를 개수로 세어 두 읽기가 같은 값을 주는지 확인하는 것이 이 실습입니다.

| Step | What you do |
|---|---|
| 1 | Read one timeline three ways, by hand, with five gaps you can add up yourself |
| 2 | Turn a rate into gaps and generate a path |
| 3 | Check the gap law, see that a probability is an area rather than a height, and watch forty minutes of waiting buy you nothing |
| 4 | Cross the bridge. `P(T₁ > t)`, `P(N(t) = 0)` and `e^(−λt)` are one number |
| 5 | Count over equal windows, compare the sample mean with the sample variance, and watch the binomial limit land on the Poisson formula |
| 6 | Look for dependence between disjoint windows, and read the result honestly |
| 7 | Merge two streams, then relabel the merged events at random |
| 8 | Optional. A rate that changes, and events that carry sizes |
| 9 | Scratch space. Change the rate and look again |

### How to use it

Two cells are marked **`TODO`**. Those two are the part worth doing by hand.
직접 채워 보는 것이 이 노트북의 핵심입니다.

> Each `TODO` is one line that reads `… = None`. Replace the `None` with your code; there is nothing to delete.
> If you run the cell before that, it stops with a `NotImplementedError` whose message says exactly this.
> **That is expected, not a broken notebook.** Each `TODO` is followed by a collapsed **Solution (정답)** cell:
> run it and the notebook continues from there. Open it only after you have tried.
> 각 `TODO` 는 `… = None` 한 줄입니다. `None` 자리에 코드를 넣으면 되고, 지울 줄은 없습니다.

> ### Before you touch anything: **File > Save a copy in Drive**
> The link I posted opens **read-only**. Colab will say *"changes will not be saved"*.
> Save your own copy first, or everything you type here disappears when you close the tab.
> 먼저 내 드라이브에 사본을 저장하세요. 안 하면 입력한 내용이 탭을 닫을 때 사라집니다.
>
> **Nothing to submit here, and no code to submit for Assignment 1 either.**
> This lab is practice. Assignment 1 (12%) went out today as a paper-based foundations checkpoint:
> six short problems, hand calculation and interpretation, one PDF on the LMS. See
> *Slide 35 — Assignment 1 Goes Out Today* and *Slide 33 — Lab Workflow*.
> What the lab gives you is a feel for the objects the assignment asks about in words.
> 이 노트북은 제출하지 않습니다. 과제 1 도 손계산과 해석으로 푸는 지면 과제이고 코드 제출은 없습니다
> (덱 *Slide 35 — Assignment 1 Goes Out Today*, *Slide 33 — Lab Workflow*).
> 이 실습은 과제가 말로 묻는 것을 숫자로 한 번 만져 보는 연습입니다.

---
## 0. Setup

Nothing to install this week. Everything below ships with Colab.
설치할 것은 없습니다. 아래에 쓰는 것은 전부 Colab 에 이미 들어 있습니다.

**시간 단위는 시간(hour) 하나로 고정합니다.** 강의에서 쓴 전화선과 같은 설정이고, `λ = 2` 는 시간당 2건입니다.
*One time unit throughout: the hour. Same phone line as the lecture, and `λ = 2` means two calls per hour.*
분으로 바꿔 넣는 순간 모든 숫자가 틀어지므로, 40분은 `2/3`, 15분은 `1/4` 로 씁니다.
*Putting minutes into `λt` breaks every number below, so forty minutes is `2/3` and fifteen minutes is `1/4`.*

그림의 축 이름은 영문으로 둡니다. 한글 축을 쓰려면 바로 아래 선택 셀을 먼저 실행하세요.
*Plot labels are kept in English. Run the optional cell below first if you want Korean labels.*

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

SEED = 42                       # 시드 고정 = 재현 가능 / fixed seed = reproducible
rng  = np.random.default_rng(SEED)

LAM = 2.0                       # 시간당 사건 수. 강의의 전화선과 같은 값 / events per hour, as in the lecture
T   = 3.0                       # 관측 구간 길이(시간) / length of the observation window, in hours

np.set_printoptions(precision=4, suppress=True)
plt.rcParams["figure.figsize"] = (11, 3.2)
plt.rcParams["axes.grid"] = True
plt.rcParams["grid.alpha"] = 0.3

# 빈칸(TODO)을 아직 안 채웠을 때 보여 줄 안내입니다. 고장이 아니라는 뜻입니다.
# Message shown when a TODO is still blank. It means the notebook is fine, not broken.
BLANK_MSG = (
    "\n\n"
    "  정상입니다. 고장이 아니라 일부러 비워 둔 칸입니다.\n"
    "     위의 None 자리에 코드를 넣고 이 셀을 다시 실행하세요. 지울 줄은 없습니다.\n"
    "     막힐 때 바로 아래 'Solution / 정답' 셀을 실행하면 이어서 진행됩니다.\n\n"
    "  This is expected, not a broken notebook.\n"
    "     Replace None above with your code and re-run. Nothing needs deleting.\n"
    "     If stuck, run the Solution cell just below and continue.\n"
)

print("numpy:", np.__version__)
print("rate lam =", LAM, "per hour,  horizon T =", T, "hours,  so lam*T =", LAM * T)

### Optional. Korean labels in plots / 그림에 한글 쓰기

Colab 의 기본 폰트에는 한글이 없어서 한글 축 이름이 네모로 깨집니다. 아래 셀을 한 번 실행하면 고쳐집니다.
필수가 아니고, 실행하면 30초쯤 걸립니다.
*Colab's default font has no Korean glyphs. Running the cell below once fixes that. It is optional and takes about half a minute.*

In [ ]:
#@title ▶ 한글 폰트 설치 / Install Korean font (optional) { display-mode: "form" }
# 실행하지 않아도 노트북은 전부 돌아갑니다. 축 이름이 영문이면 이 셀은 건너뛰세요.
# The notebook runs fine without this. Skip it if English labels are fine for you.
import subprocess, sys, matplotlib, matplotlib.font_manager as fm

subprocess.run("apt-get -qq -y install fonts-nanum > /dev/null", shell=True)
fm._load_fontmanager(try_read_cache=False)
matplotlib.rc("font", family="NanumGothic")
matplotlib.rcParams["axes.unicode_minus"] = False   # 마이너스 기호 깨짐 방지 / keep the minus sign readable
print("한글 폰트 준비 완료 / Korean font ready")

---
## 1. 하나의 타임라인, 세 가지 대상 / One timeline, three objects

먼저 손으로 읽습니다. 간격 다섯 개를 직접 정해 놓고, 나머지는 더하기만 하면 나옵니다.
*Read it by hand first. Fix five gaps yourself and everything else is addition.*

| 기호 / Symbol | 뜻 / What it is |
|---|---|
| `T_k` | k 번째 간격. 사건 k−1 에서 사건 k 까지 걸린 시간<br>The k-th gap, the time from event k−1 to event k |
| `S_k` | k 번째 도착 시각. 간격을 처음부터 더한 값<br>The k-th arrival time, the running sum of the gaps |
| `N(t)` | 시각 t 까지 일어난 사건의 수<br>The number of events that have happened by time t |

**창은 반개구간으로 셉니다.** `N(0) = 0` 이고, 정확히 시각 t 에 일어난 사건은 이미 센 것으로 봅니다.
*We count over half-open windows, so `N(0) = 0` and an event exactly at time t is already counted.*
덱 *Slide 9 — One Timeline, Three Objects* 와 같은 그림입니다.
*Same picture as Slide 9 — One Timeline, Three Objects.*

In [ ]:
# 간격을 직접 정해 둡니다. 여기서는 난수를 쓰지 않습니다.
# Five gaps written down by hand. No randomness in this cell.
gaps_toy = np.array([0.4, 0.9, 0.3, 1.1, 0.6])

# 도착 시각은 간격을 처음부터 더한 것입니다. S_k = T_1 + ... + T_k
# The arrival times are the running sums of the gaps. S_k = T_1 + ... + T_k
times_toy = np.cumsum(gaps_toy)

print("간격 T_k / gaps      ", gaps_toy)
print("도착 시각 S_k / times", times_toy)
# 기대값 / Expected  times [0.4 1.3 1.6 2.7 3.3]
#   0.4, 0.4+0.9=1.3, 1.3+0.3=1.6, 1.6+1.1=2.7, 2.7+0.6=3.3 입니다. 손으로 더해서 맞춰 보세요.
#   Add them up by hand and check. If you got [0.4 0.9 0.3 1.1 0.6] back, cumsum was not applied.

# N(t) 는 t 이하인 도착 시각의 개수입니다.
# N(t) is how many arrival times are at or below t.
for t in (1.0, 2.0, 3.0, 4.0):
    print("N(%.0f) =" % t, int(np.sum(times_toy <= t)))
# 기대값 / Expected  N(1)=1, N(2)=3, N(3)=4, N(4)=5

In [ ]:
# 위 세 대상을 한 그림으로. 위는 도착 시각, 아래는 계단으로 그린 개수.
# The three objects in one picture. Arrivals on top, the count as a staircase below.
fig, ax = plt.subplots(2, 1, sharex=True, figsize=(11, 4.2),
                       gridspec_kw={"height_ratios": [1, 2]})

ax[0].vlines(times_toy, 0, 1, color="#B4412E", lw=2)          # 사건이 일어난 순간 / the instants of the events
ax[0].hlines(0, 0, 4, color="#888888", lw=1)                  # 시간축 / the time axis
for k, s in enumerate(times_toy, 1):
    ax[0].text(s, 1.15, "S%d" % k, ha="center", color="#B4412E")
ax[0].annotate("", xy=(times_toy[1], -0.5), xytext=(times_toy[0], -0.5),
               arrowprops=dict(arrowstyle="<->", color="#2C2C2C"))
ax[0].text(times_toy[:2].mean(), -1.1, "T2", ha="center")     # 두 번째 간격만 이름을 붙인다 / only the second gap is named
ax[0].set_ylim(-1.6, 1.8); ax[0].set_yticks([]); ax[0].grid(False)
ax[0].set_title("arrival times and one gap")

grid = np.linspace(0, 4, 401)
ax[1].step(grid, [np.sum(times_toy <= t) for t in grid], where="post", color="#11241C", lw=2)
ax[1].set_xlabel("time t (hours)"); ax[1].set_ylabel("N(t)")
ax[1].set_title("the counting process")
plt.tight_layout(); plt.show()

계단이 올라가는 자리가 위의 빨간 선입니다. 같은 정보를 두 번 그린 것이지, 두 개의 과정이 아닙니다.
*The staircase steps up exactly where the red lines are. It is one piece of information drawn twice, not two processes.*

계단의 높이는 언제나 1 씩 올라갑니다. 두 사건이 정확히 같은 순간에 일어나는 일은 이 모형에 없습니다.
*Every jump has height one. Two events at the very same instant do not happen in this model.*

> 여기까지는 난수가 없었습니다. 간격을 우리가 정해 줬으니까요. 다음 절에서 그 간격을 비율 하나로부터 뽑습니다.

---
## 2. 비율에서 경로 하나로 / From a rate to a path

간격을 우리가 정하는 대신, 모형에서 뽑습니다.

**여기서 방향이 강의와 반대입니다.** 강의에서는 포아송 개수에서 출발해 첫 대기시간이 지수분포임을
유도했습니다. 이 실습은 반대로 갑니다. 독립인 지수 간격을 먼저 만들고 누적해서, 창 안의 개수가
포아송으로 나타나는지 확인합니다. 두 방향이 같은 하나의 모형을 가리키는지를 눈으로 보는 것입니다.
*The direction here is the reverse of the lecture's. There we started from Poisson counts and derived that the
first waiting time is exponential. Here we generate independent exponential gaps, accumulate them, and check
whether Poisson counts appear. The point is to see that the two directions describe one model.*

**비율만으로는 지수분포가 나오지 않습니다.** 비율은 눈금이고, 분포가 되려면 모형을 하나 골라야 합니다
(덱 *Slide 2 — How Soon Is the Next Call?* 의 마지막 줄, *Slide 16 — When Is a Counting Process Poisson?*).
아래에서 간격을 지수분포로 뽑는 것은 그 모형을 **가정하는** 것이지 비율에서 따라 나오는 것이 아닙니다.
*A rate alone does not give a distribution. A rate is a scale; it becomes a distribution only once a model is
chosen (Slide 2 — How Soon Is the Next Call?, and Slide 16 — When Is a Counting Process Poisson?). Drawing the
gaps as exponential below **assumes** that model rather than deriving it from the rate.*

덱 *Slide 19 — The Exponential Gap* 과 *Slide 22 — The Mean Wait* 에서 나온 세 줄 중
여기서 쓰는 것은 마지막 줄입니다.
*Three lines from Slide 19 — The Exponential Gap and Slide 22 — The Mean Wait. We use the last one here.*

```
f(t) = λ e^(−λt)        밀도 / density
P(T > t) = e^(−λt)      생존함수 / survival
E[T] = 1/λ              평균 / mean          <- 이번 빈칸이 묻는 것
```

**`λ` 는 확률이 아닙니다.** 단위가 「1 / 시간」이고, 평균 간격은 그 역수입니다. 시간당 2건이면 평균 간격은 30분입니다.
*`λ` is not a probability. Its unit is one over time, and the mean gap is its reciprocal. Two per hour means a mean gap of thirty minutes.*

### 이 노트북에서 쓰는 NumPy / NumPy used in this notebook

파이썬이 처음이어도 따라올 수 있도록, 이 노트북에 나오는 것만 모아 둡니다. 외울 필요는 없고 막힐 때 돌아와 보면 됩니다.
*Everything below is what this notebook actually uses, nothing more. No need to memorize it, just come back when you are stuck.*

| 쓰는 것 / Code | 뜻 / What it means |
|---|---|
| `np.cumsum(v)` | 벡터 `v` 를 처음부터 누적해서 더한 벡터<br>The running sums of `v` |
| `np.array(list)` | 파이썬 리스트를 NumPy 배열로 바꾼다<br>Turns a Python list into a NumPy array |
| `arr[mask]` | `mask` 가 `True` 인 자리만 골라낸 배열<br>The entries of `arr` where `mask` is `True` |
| `np.sum(mask)` | `True` 의 개수<br>How many entries are `True` |
| `np.exp(x)` | e 의 x 제곱<br>e raised to the power x |
| `arr.mean()`, `arr.var(ddof=1)` | 표본 평균과 표본 분산<br>The sample mean and the sample variance |
| `np.random.default_rng(SEED)` | 난수 생성기 `rng` 를 만든다. 같은 `SEED` 는 같은 난수열을 재현한다<br>Creates the generator `rng`. The same `SEED` reproduces the same numbers |

**`rng.exponential(scale=b)`**

평균이 `b` 인 지수분포에서 값을 하나 뽑습니다. 인자 이름이 `rate` 가 아니라 **`scale`**, 즉 **평균**이라는 점이 중요합니다.
*Draws one value from an exponential distribution whose **mean** is `b`. The argument is `scale`, the mean, not the rate.*

```python
rng.exponential(scale=0.25)
# 평균이 0.25 인 지수분포에서 한 번 뽑는다
# draws once from an exponential distribution with mean 0.25
```

아래 TODO 에서 생각할 것은 하나입니다. **시간당 `lam` 건이면 간격 하나의 평균은 얼마인가.**
*There is one thing to work out in the TODO below. **If events come at `lam` per hour, what is the mean of one gap?***

In [ ]:
# ---------------------------------------------------------------- TODO 1 ----
# 다음 이름을 씁니다 / Names used below
#   lam      : 시간당 사건 수. 비율 / events per hour, the rate
#   T        : 관측 구간의 길이. 이 시각을 넘으면 멈춘다
#              length of the observation window. We stop once the clock passes it
#   rng      : 난수 생성기 / random number generator
#   mean_gap : 간격 하나의 평균   <- 여기가 빈칸
#              the mean of one gap   <- this is the blank
#   t        : 지금까지 흘러온 시각 / the clock, in hours since the start
#   out      : 구간 안에 들어온 도착 시각을 모으는 리스트
#              list collecting the arrival times that fit inside the window
#
# 참고 / Reference   rng.exponential(scale=b)
#   평균이 b 인 지수분포에서 값을 하나 뽑는다.
#   Draws one value from an exponential distribution with mean b.
#   예 / e.g.   rng.exponential(scale=0.25)  ->  평균 0.25 인 지수분포에서 한 번
#                                                one draw, mean 0.25

def arrival_times(lam, T, rng):
    """Return the arrival times of a Poisson process of rate lam inside (0, T].

    The gaps are exponential. Adding them up gives the arrival times.
    """
    out = []               # 도착 시각을 모을 리스트 / collects the arrival times
    t = 0.0                # 시계를 0 에서 시작한다 / the clock starts at zero

    while True:
        # TODO 1. 사건이 시간당 lam 건의 비율로 일어납니다. 간격 하나의 평균은 얼마인가요?
        #         그 값을 mean_gap 에 넣으세요. 한 줄입니다.
        # TODO 1. Events arrive at a rate of lam per hour. What is the mean of one gap?
        #         Put that value into mean_gap. One line.
        mean_gap = None
        if mean_gap is None:
            raise NotImplementedError(BLANK_MSG)

        t = t + rng.exponential(scale=mean_gap)   # 간격을 하나 뽑아 시계를 앞으로 옮긴다 / draw one gap, move the clock
        if t > T:                                 # 구간을 넘었으면 멈춘다 / stop once we pass the horizon
            break
        out.append(t)                             # 구간 안에 들어온 사건만 남긴다 / keep the events that fit

    return np.array(out)
# --------------------------------------------------------------------------


paths = [arrival_times(LAM, T, np.random.default_rng(SEED + i)) for i in range(2000)]
counts_per_path = np.array([len(p) for p in paths])

print("첫 경로의 도착 시각 / arrival times of path 1 :", np.round(paths[0], 3))
print("다섯 경로의 사건 수 / event counts, paths 1-5 :", counts_per_path[:5])
print("2000 경로의 평균    / mean over 2000 paths    :", round(counts_per_path.mean(), 4))
print("이론값 lam*T        / theory lam*T            :", LAM * T)
# 기대값 / Expected  경로 1-5 의 사건 수 [2 9 7 9 7], 2000 경로 평균 6.0315
#   시드를 고정했으므로 이 값이 그대로 나와야 합니다.
#   The seed is fixed, so you should see exactly these numbers.
#   첫 경로는 2건이고 둘째는 9건입니다. 평균이 6이어도 경로 하나는 이렇게 흔들립니다.
#   The first path has 2 events and the second has 9. The mean is 6, but a single path swings like this.
#   평균이 3 근처로 나왔다면 mean_gap 에 1/lam 대신 lam 을 넣은 것입니다.
#   A mean near 3 means lam went into mean_gap instead of 1/lam.

In [ ]:
#@title ▶ Solution / 정답. 막혔을 때만 실행하세요 (위에서 쓴 것을 덮어씁니다) { display-mode: "form" }
# 위 TODO 셀에서 빈칸 한 줄만 채운 것입니다. 실행 로직은 빈칸 한 줄만 다릅니다.
# The TODO cell with the blank filled in. Only that one line of logic differs.

def arrival_times(lam, T, rng):
    """Return the arrival times of a Poisson process of rate lam inside (0, T].

    The gaps are exponential. Adding them up gives the arrival times.
    """
    out = []               # 도착 시각을 모을 리스트 / collects the arrival times
    t = 0.0                # 시계를 0 에서 시작한다 / the clock starts at zero

    while True:
        mean_gap = 1.0 / lam         # 빈칸이었던 줄. 시간당 lam 건이면 평균 간격은 그 역수다
                                     # the blank. At lam per hour the mean gap is one over lam
        if mean_gap is None:
            raise NotImplementedError(BLANK_MSG)

        t = t + rng.exponential(scale=mean_gap)   # 간격을 하나 뽑아 시계를 앞으로 옮긴다 / draw one gap, move the clock
        if t > T:                                 # 구간을 넘었으면 멈춘다 / stop once we pass the horizon
            break
        out.append(t)                             # 구간 안에 들어온 사건만 남긴다 / keep the events that fit

    return np.array(out)


paths = [arrival_times(LAM, T, np.random.default_rng(SEED + i)) for i in range(2000)]
counts_per_path = np.array([len(p) for p in paths])

print("첫 경로의 도착 시각 / arrival times of path 1 :", np.round(paths[0], 3))
print("다섯 경로의 사건 수 / event counts, paths 1-5 :", counts_per_path[:5])
print("2000 경로의 평균    / mean over 2000 paths    :", round(counts_per_path.mean(), 4))
print("이론값 lam*T        / theory lam*T            :", LAM * T)
# 기대값 / Expected  경로 1-5 의 사건 수 [2 9 7 9 7], 2000 경로 평균 6.0315
#   시드를 고정했으므로 이 값이 그대로 나와야 합니다.
#   The seed is fixed, so you should see exactly these numbers.
#   첫 경로는 2건이고 둘째는 9건입니다. 평균이 6이어도 경로 하나는 이렇게 흔들립니다.
#   The first path has 2 events and the second has 9. The mean is 6, but a single path swings like this.
#   평균이 3 근처로 나왔다면 mean_gap 에 1/lam 대신 lam 을 넣은 것입니다.
#   A mean near 3 means lam went into mean_gap instead of 1/lam.

---
## 3. 간격의 법칙과 무기억성 / The gap law, and lack of memory

이제 간격만 잔뜩 뽑아 분포를 확인합니다. 경로를 만들 필요는 없습니다. 간격 하나하나가 독립이니까요.
*Now we just draw many gaps and look at their distribution. No path needed, because the gaps are independent of each other.*

이 절에서 확인하는 것은 둘입니다.
1. 간격의 히스토그램이 밀도 `λe^(−λt)` 와 겹치는가
2. **이미 기다린 시간이 남은 대기에 영향을 주는가** — *Slide 2 — How Soon Is the Next Call?* 의 사십 분 문제

In [ ]:
# 간격 20만 개. 한 번에 뽑습니다.
# Two hundred thousand gaps, drawn in one go.
gaps = np.random.default_rng(SEED).exponential(scale=1.0 / LAM, size=200_000)

print("표본 평균 간격 / sample mean gap :", round(gaps.mean(), 4), "시간 / hours",
      "=", round(gaps.mean() * 60, 2), "분 / minutes")
print("이론 1/lam    / theory 1/lam    :", 1.0 / LAM, "시간 / hours = 30.0 분 / minutes")
# 기대값 / Expected  표본 평균 0.4995 시간 = 29.97 분

edges = np.linspace(0, 2.5, 60)
plt.hist(gaps, bins=edges, density=True, color="#86A996", edgecolor="white", label="simulated gaps")
grid = np.linspace(0, 2.5, 300)
plt.plot(grid, LAM * np.exp(-LAM * grid), color="#B4412E", lw=2, label="density lam*exp(-lam*t)")
plt.xlabel("gap length (hours)"); plt.ylabel("density"); plt.legend(); plt.show()
# 막대가 빨간 곡선을 따라가면 맞은 것입니다. 오른쪽 꼬리는 표본이 적어 들쭉날쭉합니다.
# The bars should track the red curve. The right tail is ragged because few samples land there.

### 높이가 아니라 면적이 확률입니다 / The area is the probability, not the height

덱 *Slide 21 — Probability Density Function to Cumulative Distribution Function* 의 아래 줄을 숫자로 확인합니다.
밀도 `f(t)` 는 높이이고, 확률은 그 곡선 **아래의 면적**이며, 누적분포 `F(t)` 는 0 부터 t 까지 쌓은 면적입니다.
*Checking the bottom row of Slide 21 in numbers. The density `f(t)` is a height, a probability is the **area** under
that curve, and the CDF `F(t)` is the area accumulated from 0 to t.*

구간 `(a, b]` 하나를 잡아 세 가지를 비교합니다.

| 무엇 / What | 어떻게 / How |
|---|---|
| 누적분포의 차 / from the CDF | `F(b) − F(a)`, 여기서 `F(t) = 1 − e^(−λt)`<br>`F(b) − F(a)`, with `F(t) = 1 − e^(−λt)` |
| 표본의 비율 / from the sample | 위에서 뽑아 둔 간격 20만 개 중 그 구간에 떨어진 비율<br>The fraction of the 200,000 gaps that landed in the interval |
| 밀도의 높이 / the density height | `f(a)`, `f(b)`. 확률이 아니고 단위가 「1 / 시간」입니다<br>`f(a)`, `f(b)`. Not probabilities, and their unit is one over time |

**`np.exp(x)`** — e 의 x 제곱입니다. / *e raised to the power x.*
**`plt.fill_between(x, y)`** — 곡선 `y` 와 0 사이를 색으로 채웁니다. 면적을 눈으로 보려고 씁니다.
*Fills the region between the curve `y` and zero. We use it to make the area visible.*

In [ ]:
a, b = 0.25, 0.75          # 구간 (a, b]. 15분에서 45분 / the interval, fifteen to forty-five minutes

def F(t):
    """누적분포 F(t) = P(T <= t) = 1 - e^(-lam*t) / the CDF."""
    return 1.0 - np.exp(-LAM * t)

area  = F(b) - F(a)                            # 면적, 즉 확률 / the area, which is the probability
share = np.mean((gaps > a) & (gaps <= b))      # 표본에서 그 구간에 떨어진 비율 / the sample fraction there

print("누적분포의 차 F(b) - F(a) / from the CDF      :", round(area, 4))
print("표본에서의 비율            / from the sample   :", round(share, 4))
print("밀도의 높이 f(a), f(b)     / density heights   :",
      round(LAM * np.exp(-LAM * a), 4), ",", round(LAM * np.exp(-LAM * b), 4))
# 기대값 / Expected  F(b) - F(a) = 0.3834, 표본 비율 0.3822, 높이 1.2131 과 0.4463
#   앞의 두 값이 소수 둘째 자리까지 겹칩니다. 면적이 곧 확률이라는 뜻입니다.
#   The first two agree to two decimals. That is what "the area is the probability" means.
#   f(a) = 1.2131 은 1 을 넘습니다. 밀도는 확률이 아니므로 1 을 넘어도 됩니다.
#   f(a) = 1.2131 exceeds one. A density is not a probability, so it may.
#   0.3834 대신 0.7769 가 나왔다면 F(a) 를 빼지 않고 F(b) 만 쓴 것입니다.
#   Getting 0.7769 instead means F(a) was not subtracted.

grid = np.linspace(0, 2.5, 300)
dens = LAM * np.exp(-LAM * grid)
plt.plot(grid, dens, color="#B4412E", lw=2, label="density lam*exp(-lam*t)")
inside = (grid >= a) & (grid <= b)
plt.fill_between(grid[inside], dens[inside], color="#BFD4C6",
                 label="area from a to b = F(b) - F(a)")
plt.vlines([a, b], 0, [LAM * np.exp(-LAM * a), LAM * np.exp(-LAM * b)],
           color="#2C2C2C", lw=1, linestyles="dashed")
plt.xlabel("gap length (hours)"); plt.ylabel("density"); plt.legend(); plt.show()
# 색칠된 부분의 넓이가 위에서 계산한 0.3834 입니다. 곡선의 높이가 아니라 넓이입니다.
# The shaded region has area 0.3834, the number printed above. The area, not the height.

### 사십 분을 기다렸습니다 / Forty minutes have passed

*Slide 2 — How Soon Is the Next Call?* 의 질문입니다. 마지막 전화로부터 40분이 지났습니다.
다음 전화가 이제 더 빨리 올까요?
*The question from Slide 2 — How Soon Is the Next Call? Forty minutes have passed since the last call. Is the next one now more likely to arrive soon?*

확인하는 방법은 단순합니다. 뽑아 둔 간격 중 **40분을 넘긴 것만** 골라내고, 거기서 이미 지난 40분을 빼면
그것이 「그 시점에서 본 남은 대기」입니다. 그 분포를 처음부터 잰 대기와 비교합니다.
*Keep only the gaps that already exceeded forty minutes, subtract the forty minutes that have passed,
and what remains is the wait seen from that moment. Compare its distribution with a wait measured from scratch.*

In [ ]:
s_past  = 2.0 / 3.0     # 이미 지난 40분 / the forty minutes already elapsed, in hours
t_extra = 1.0 / 4.0     # 앞으로 더 기다릴 15분 / fifteen more minutes, in hours

survived  = gaps[gaps > s_past]          # 40분을 넘긴 간격만 / the gaps that lasted past forty minutes
remaining = survived - s_past            # 그 시점에서 본 남은 대기 / the wait still to come, seen from that moment

print("40분을 넘긴 경우의 수 / gaps that survived forty minutes :", len(survived))
print()
print("조건부. 남은 대기가 15분을 더 넘을 비율 / conditional, remaining wait exceeds 15 more minutes :",
      round(np.mean(remaining > t_extra), 4))
print("무조건. 처음부터 15분을 넘을 비율      / unconditional, a fresh wait exceeds 15 minutes      :",
      round(np.mean(gaps > t_extra), 4))
print("이론 e^(-lam*0.25) = e^(-0.5)          / theory                                              :",
      round(np.exp(-LAM * t_extra), 4))
print()
print("조건부 평균 남은 대기 / conditional mean remaining wait :",
      round(remaining.mean() * 60, 2), "분 / minutes")
print("이론 1/lam           / theory 1/lam                    :", round(60 / LAM, 2), "분 / minutes")
# 기대값 / Expected  조건부 0.606, 무조건 0.6053, 이론 0.6065, 조건부 평균 30.03 분
#   세 숫자가 소수 둘째 자리까지 겹칩니다. 40분을 기다린 것이 아무것도 바꾸지 않았다는 뜻입니다.
#   The three numbers agree to two decimals. The forty minutes changed nothing.
#   조건부 값만 크게 작아졌다면 remaining 에서 s_past 를 빼지 않은 것입니다.
#   A much smaller conditional value means s_past was not subtracted from remaining.

세 숫자가 겹칩니다. 40분을 기다린 것이 남은 대기에 아무것도 해 주지 않았습니다.
평균 남은 대기도 여전히 30분이므로, 처음부터 세면 총 70분을 기다리게 됩니다. 덱 *Slide 24 — Worked Example 2. The Forty Minute Wait* 의 손계산과 같은 값입니다.
*The three numbers agree. The forty minutes bought nothing. The mean remaining wait is still thirty minutes,
so the expected total from the start is seventy. Same numbers as Slide 24 — Worked Example 2.*

> **이것은 모형의 성질이지 줄 서기의 법칙이 아닙니다.** 30분마다 정확히 오는 버스를 25분 기다렸다면
> 남은 대기는 5분에 가깝지 30분이 아닙니다. *Slide 25 — When Memorylessness Fails* 가 그 이야기입니다.
> *This is a property of one model, not a law of real queues. A bus scheduled every thirty minutes,
> waited on for twenty-five, has about five minutes left, not thirty.*
>
> 반례로 정규분포를 들지 마세요. 음수가 나올 수 있어서 대기시간이 아닙니다.
> *Do not reach for a normal distribution as the counterexample. It can go negative, so it is not a waiting time.*

---
## 4. 다리. 개수와 간격은 같은 질문 / The bridge

덱 *Slide 18 — From Zero Calls to the First Waiting Time* 이 오늘의 중심입니다.
*Slide 18 — From Zero Calls to the First Waiting Time is the centre of today.*

```
P(T₁ > t)  =  P(N(t) = 0)  =  e^(−λt)
```

왼쪽은 **간격**의 말입니다. 첫 사건까지의 대기가 t 를 넘는다.
오른쪽 가운데는 **개수**의 말입니다. t 까지 일어난 사건이 0건이다.
둘은 같은 사건을 두 가지 언어로 쓴 것입니다.
*The left is the language of gaps, the middle the language of counts. They describe the same event.*

여기서는 그 값이 실제로 `e^(−λt)` 인지를 **서로 다른 두 난수 표본으로** 확인합니다.

| 어느 쪽 / Which side | 어떻게 재나 / How we measure it |
|---|---|
| 간격 쪽 / gaps | 지수분포에서 첫 간격을 20만 개 뽑아, t 를 넘는 비율을 센다<br>Draw 200,000 first gaps and count the fraction exceeding t |
| 개수 쪽 / counts | `Pois(λt)` 에서 개수를 20만 개 뽑아, 0 인 비율을 센다<br>Draw 200,000 counts from `Pois(λt)` and count the fraction equal to zero |

두 줄은 서로 다른 생성기를 쓰므로, 값이 겹치는 것은 정의 때문이 아니라 **분포가 맞물려 있기 때문**입니다.
*The two rows use different generators, so agreement is not a matter of definition but of the two laws fitting together.*

> 다만 이것은 **두 확률법칙이 같은 값을 준다는 수치 확인**입니다. 한 표본경로에서
> `T₁ > t` 와 `N(t) = 0` 이 같은 사건임을 직접 보이는 것도, 그 등식을 증명하는 것도 아닙니다.
> 증명은 강의의 *Slide 18 — From Zero Calls to the First Waiting Time* 쪽입니다.
> *This is a numerical check that the two laws agree. It neither exhibits the two events coinciding on one
> sample path nor proves the identity. The proof is on Slide 18 — From Zero Calls to the First Waiting Time.*

**`rng.poisson(mu, size=R)`** — 평균이 `mu` 인 포아송분포에서 `R` 개를 뽑습니다.
*Draws `R` values from a Poisson distribution with mean `mu`.*

아래 TODO 에서 생각할 것은 하나입니다. **t0 까지 한 건도 없을 확률을 `lam` 과 `t0` 로 어떻게 쓰는가.**

In [ ]:
# ---------------------------------------------------------------- TODO 2 ----
# 다음 이름을 씁니다 / Names used below
#   LAM       : 시간당 사건 수 / events per hour
#   t0        : 우리가 보는 시각. 1시간 / the time we look at, one hour
#   p_theory  : t0 까지 사건이 한 건도 없을 확률   <- 여기가 빈칸
#               probability of no event by time t0   <- this is the blank
#   first_gap : 첫 간격 20만 개 / two hundred thousand first gaps
#   n_counts  : Pois(LAM*t0) 에서 뽑은 개수 20만 개 / two hundred thousand counts from Pois(LAM*t0)
#
# 참고 / Reference   np.exp(x)
#   e 의 x 제곱을 돌려준다. x 가 배열이면 원소마다 계산한다.
#   Returns e to the power x, elementwise if x is an array.
#   예 / e.g.   np.exp(-1.0)  ->  0.3679, that is one over e

t0 = 1.0

# TODO 2. 시각 t0 까지 사건이 한 건도 없을 확률을 LAM 과 t0 로 쓰면 무엇인가요?
#         덱 Slide 19 — The Exponential Gap 의 생존함수 P(T > t) = e^(-lam*t) 를 그대로 쓰면 됩니다. 한 줄입니다.
# TODO 2. Write the probability of no event by time t0, in terms of LAM and t0.
#         The survival function P(T > t) = e^(-lam*t) from Slide 19 is all you need. One line.
p_theory = None
if p_theory is None:
    raise NotImplementedError(BLANK_MSG)

# 간격 쪽. 첫 간격이 t0 를 넘는 비율 / gaps side, the fraction of first gaps exceeding t0
first_gap = np.random.default_rng(SEED).exponential(scale=1.0 / LAM, size=200_000)

# 개수 쪽. 다른 생성기로 개수를 뽑아 0 인 비율 / counts side, a different generator, fraction equal to zero
n_counts = np.random.default_rng(SEED + 1).poisson(LAM * t0, size=200_000)

print("이론   e^(-lam*t0)                  / theory              :", round(p_theory, 4))
print("간격 쪽. 첫 간격 > t0 인 비율       / gaps side            :", round(np.mean(first_gap > t0), 4))
print("개수 쪽. N(t0) = 0 인 비율          / counts side          :", round(np.mean(n_counts == 0), 4))
# 기대값 / Expected  이론 0.1353, 간격 쪽 0.1349, 개수 쪽 0.1345
#   셋이 소수 둘째 자리까지 겹칩니다. 다리가 놓인 것입니다.
#   The three agree to two decimals. That is the bridge.
#   0.8647 이 나왔다면 e^(-lam*t0) 대신 1 - e^(-lam*t0) 를 쓴 것입니다. 그것은 「적어도 한 건」입니다.
#   Getting 0.8647 means 1 - e^(-lam*t0) was used. That is the probability of at least one event.
# --------------------------------------------------------------------------

In [ ]:
#@title ▶ Solution / 정답. 막혔을 때만 실행하세요 (위에서 쓴 것을 덮어씁니다) { display-mode: "form" }
# 위 TODO 셀에서 빈칸 한 줄만 채운 것입니다. 실행 로직은 빈칸 한 줄만 다릅니다.
# The TODO cell with the blank filled in. Only that one line of logic differs.

t0 = 1.0
p_theory = np.exp(-LAM * t0)   # 빈칸이었던 줄. 생존함수를 그대로 쓴 것이다
                               # the blank. This is the survival function, written out
if p_theory is None:
    raise NotImplementedError(BLANK_MSG)

# 간격 쪽. 첫 간격이 t0 를 넘는 비율 / gaps side, the fraction of first gaps exceeding t0
first_gap = np.random.default_rng(SEED).exponential(scale=1.0 / LAM, size=200_000)

# 개수 쪽. 다른 생성기로 개수를 뽑아 0 인 비율 / counts side, a different generator, fraction equal to zero
n_counts = np.random.default_rng(SEED + 1).poisson(LAM * t0, size=200_000)

print("이론   e^(-lam*t0)                  / theory              :", round(p_theory, 4))
print("간격 쪽. 첫 간격 > t0 인 비율       / gaps side            :", round(np.mean(first_gap > t0), 4))
print("개수 쪽. N(t0) = 0 인 비율          / counts side          :", round(np.mean(n_counts == 0), 4))
# 기대값 / Expected  이론 0.1353, 간격 쪽 0.1349, 개수 쪽 0.1345
#   셋이 소수 둘째 자리까지 겹칩니다. 다리가 놓인 것입니다.
#   The three agree to two decimals. That is the bridge.
#   0.8647 이 나왔다면 e^(-lam*t0) 대신 1 - e^(-lam*t0) 를 쓴 것입니다. 그것은 「적어도 한 건」입니다.
#   Getting 0.8647 means 1 - e^(-lam*t0) was used. That is the probability of at least one event.

세 숫자가 겹칩니다. 「아직 첫 사건을 기다리는 중」과 「지금까지 0건」이 같은 말이라는 것이 확인됐습니다.

**다만 이 등식 하나가 모든 것을 증명하지는 않습니다.** 첫 간격이 지수분포라는 것만으로는
*모든* 간격이 지수분포이고 서로 독립이라는 결론이 나오지 않습니다. 그 결론은 독립증분과 정상증분에서 옵니다.
덱 *Slide 26 — Gaps to Epochs to Counts* 의 오른쪽 칸이 이것입니다.
*This one identity does not prove everything. That the first gap is exponential does not by itself make
every gap exponential and independent. That comes from independent and stationary increments.*

---
## 5. 창 안의 개수는 포아송 / The count over a window

이제 §2 에서 만든 `arrival_times` 를 **그대로 갖다 씁니다.** 새로 짜지 않습니다.
*We now reuse `arrival_times` from section 2 as it is. Nothing gets rewritten.*

긴 경로를 하나 만들고 길이 3시간짜리 창 2000개로 자릅니다. 창마다 사건 수를 세면 `Pois(λ·3) = Pois(6)` 입니다.
*One long path, chopped into 2000 windows of three hours. The count in each window should be `Pois(6)`.*

**`np.bincount(idx, minlength=W)`** — `idx` 안에 0, 1, 2, … 가 각각 몇 번 나오는지 센 벡터를 돌려줍니다.
*Counts how many times each of 0, 1, 2, … appears in `idx`.*
**`np.floor(x)`** — `x` 를 내림한 값입니다. 사건 시각을 창 길이로 나눠 내림하면 그 사건이 몇 번째 창인지 나옵니다.
*Rounds `x` down. Dividing an event time by the window length and rounding down gives the index of its window.*

In [ ]:
h, W = 3.0, 2000                       # 창 길이 3시간, 창 2000개 / window length and number of windows
long_path = arrival_times(LAM, h * W, np.random.default_rng(SEED))
which_window = np.floor(long_path / h).astype(int)     # 각 사건이 몇 번째 창인지 / the window index of each event
counts = np.bincount(which_window, minlength=W)        # 창마다 사건 수 / the count in each window

print("창의 개수 / number of windows :", W)
print("표본 평균 / sample mean       :", round(counts.mean(), 4), "  이론 lam*h / theory :", LAM * h)
print("표본 분산 / sample variance   :", round(counts.var(ddof=1), 4), "  이론 lam*h / theory :", LAM * h)
print("분산/평균 / variance over mean:", round(counts.var(ddof=1) / counts.mean(), 4))
# 기대값 / Expected  평균 6.02, 분산 6.1867, 비 1.0277
#   평균과 분산이 둘 다 6 근처입니다. 포아송의 표시입니다.
#   Both the mean and the variance sit near 6. That is the Poisson signature.
#   비가 1 에서 얼마나 떨어져야 이상한지는 창의 개수가 정합니다. 여기서는 대략 sqrt(2/W) = 0.032 규모입니다.
#   How far the ratio may sit from 1 is set by the number of windows, here about sqrt(2/W) = 0.032.

k = np.arange(0, 17)
from math import factorial
pmf = np.exp(-LAM * h) * (LAM * h) ** k / np.array([factorial(int(i)) for i in k])
plt.bar(k, np.bincount(counts, minlength=17)[:17] / W, color="#86A996",
        edgecolor="white", label="simulated windows")
plt.plot(k, pmf, "o-", color="#B4412E", lw=2, label="Poisson(6) pmf")
plt.xlabel("events per 3-hour window"); plt.ylabel("proportion"); plt.legend(); plt.show()

### 강의의 조각 그림을 숫자로 / The slicing picture, in numbers

위에서는 지수 간격을 쌓아 개수가 포아송이 되는 것을 보았습니다. 강의는 같은 결론에 **다른 길**로 갔습니다.
덱 *Slide 12 — Slice Time into Pieces* 에서 구간을 `n` 조각으로 자르고, *Slide 13 — The Binomial Law for n Slices*
에서 조각당 확률 `p = λt/n` 의 이항분포를 세운 뒤, *Slide 14 — From Slices to the Poisson Formula* 에서 `n → ∞`
극한을 보냈습니다.
*Above we stacked exponential gaps and watched the counts come out Poisson. The lecture reached the same place by a
different road: cut the window into `n` slices (Slide 12), write the binomial law with `p = λt/n` (Slide 13), and
send `n → ∞` (Slide 14).*

그 극한을 숫자로 확인합니다. 3시간 창에 **정확히 6건**일 확률 하나만 봅니다.
*We check that limit numerically, on one probability: exactly six events in a three-hour window.*

**`math.comb(n, k)`** — 이항계수. n 개에서 k 개를 고르는 경우의 수입니다.
*The binomial coefficient, the number of ways to choose k out of n.*
**`math.factorial(k)`** — k 의 계승 `k!` 입니다. / *The factorial `k!`.*

In [ ]:
from math import comb, factorial

t_win, k_look = 3.0, 6               # 창 길이 3시간, 「정확히 6건」 / a three-hour window, exactly six events
mu = LAM * t_win                     # 포아송 모수 lam*t = 6 / the Poisson parameter

pois = np.exp(-mu) * mu ** k_look / factorial(k_look)    # 포아송 확률 / the Poisson probability

print("3시간 창에 정확히 6건일 확률 / probability of exactly 6 events in a 3-hour window")
print("  조각 수 n / slices   이항 Binomial(n, lam*t/n)   포아송 Pois(lam*t)      차 / difference")
for n in (10, 50, 200, 1000, 10000):
    p = mu / n                                           # 조각당 확률 / probability per slice
    binom = comb(n, k_look) * p ** k_look * (1 - p) ** (n - k_look)
    print("  %8d           %.6f                   %.6f            %+.6f" % (n, binom, pois, binom - pois))
# 기대값 / Expected  n=10 은 0.250823, n=50 은 0.171186, n=200 은 0.163086
#   n=1000 gives 0.161107, n=10000 gives 0.160671, and the Poisson value is 0.160623
#   n 을 키우면 이항 확률이 포아송 확률로 내려옵니다. 덱 14장의 극한이 이것입니다.
#   As n grows the binomial probability settles onto the Poisson one. That is the limit on Slide 14.
#   n = 10 에서는 조각당 확률이 p = 0.6 이라 「조각 하나에 사건 하나」 가정이 무리입니다. 그래서 차가 큽니다.
#   At n = 10 each slice carries p = 0.6, so "at most one event per slice" is a poor assumption. Hence the gap.
#   포아송 값이 n 에 따라 변한다면 mu 에 lam 만 넣고 t_win 을 빼먹은 것입니다.
#   A Poisson value that changes with n means t_win was left out of mu.

막대와 점이 겹칩니다. 간격을 지수분포로 뽑기만 했는데 개수가 포아송이 되었습니다. 우리가 넣은 적이 없는 성질입니다.
*The bars and the dots line up. We only drew exponential gaps, and the counts came out Poisson. We never put that in.*

> **평균과 분산이 가까운 것은 포아송과 어긋나지 않는다는 근거이지, 포아송임을 세우는 충분조건이 아닙니다.**
> 포아송이면 반드시 그렇지만, 그런 성질을 가진 다른 과정도 있습니다.
> 덱 *Slide 32 — Does a Poisson Model Fit the Data?* 의 첫 번째 칸이 이 이야기입니다.
> *Consistent with Poisson, but not sufficient to establish it. Every Poisson count has equal mean and
> variance, and so do some other processes.*

---
## 6. 독립증분과 정상증분 / Independent and stationary increments

정의의 나머지 두 조건을 같은 자료로 봅니다.

- **정상증분 / stationary increments** — 같은 길이의 창이면 어디에 놓여 있든 개수의 분포가 같다
- **독립증분 / independent increments** — 겹치지 않는 창의 개수가 서로 독립이다

**`np.corrcoef(a, b)[0, 1]`** — 두 벡터의 표본 상관계수 하나를 돌려줍니다.
*Returns the single sample correlation between two vectors.*

In [ ]:
# 정상증분. 창을 앞쪽 절반과 뒤쪽 절반으로 갈라 평균과 분산을 비교한다.
# Stationary increments. Split the windows into an early half and a late half and compare.
early, late = counts[:W // 2], counts[W // 2:]
print("앞쪽 1000창 / first 1000 windows : 평균 %.4f / mean,  분산 %.4f / variance"
      % (early.mean(), early.var(ddof=1)))
print("뒤쪽 1000창 / last  1000 windows : 평균 %.4f / mean,  분산 %.4f / variance"
      % (late.mean(), late.var(ddof=1)))
print()

# 독립증분. 이웃한 두 창의 개수 사이 상관.
# Independent increments. The correlation between the counts in neighbouring windows.
a, b = counts[0::2], counts[1::2]
r = np.corrcoef(a, b)[0, 1]
print("이웃 창의 상관 / correlation between neighbouring windows :", round(r, 4))
print("0 과 구별되지 않는 크기 규모 1/sqrt(%d) / the scale of pure noise :" % len(a),
      round(1 / np.sqrt(len(a)), 4))
# 기대값 / Expected  앞쪽 평균 6.0990, 뒤쪽 평균 5.9410, 상관 -0.0333, 잡음 규모 0.0316
#   두 평균의 차이 0.158 은 창 1000개에서 나올 만한 크기입니다. 평균 하나의 흔들림이 sqrt(6/1000) = 0.077 이니까요.
#   The gap of 0.158 between the two means is ordinary for 1000 windows, since one mean wobbles by sqrt(6/1000) = 0.077.
#   상관의 크기도 잡음 규모와 비슷합니다. 독립과 어긋나지 않는다는 증거로 읽기에 충분합니다.
#   The correlation is about the size of pure noise, which is what independence would produce.
#   0.3 이나 0.5 처럼 나왔다면 같은 창을 두 번 넣었는지 확인하세요.
#   Something like 0.3 or 0.5 means the same window probably went in twice.

> **상관이 0 인 것과 독립인 것은 다릅니다.** 독립증분은 무상관보다 훨씬 강한 주장입니다.
> 위 숫자는 독립과 어긋나지 않는다는 증거이지 독립의 증명이 아닙니다.
> 덱 *Slide 32 — Does a Poisson Model Fit the Data?* 의 두 번째 함정입니다.
> *Zero correlation is not independence. Independent increments is a much stronger statement than uncorrelated
> increments. The number above is evidence consistent with independence, not a proof of it.*

### 정상증분인데 정상과정은 아닙니다 / Stationary increments, not a stationary process

2주차의 정상성과 헷갈리기 쉬운 자리입니다. 같은 길이의 창은 어디에 놓여도 분포가 같지만,
`N(t)` 자체는 t 가 커질수록 평균이 커집니다.
덱 *Slide 27 — Stationary Increments, Not a Stationary Process* 가 이 구분입니다.
*Easy to confuse with week 2. Equal-length windows have the same distribution wherever they sit, but `N(t)` itself
has a mean that grows with t.*

In [ ]:
# 같은 창 안에서 시각 t 까지의 개수를 재면 t 에 따라 평균이 커진다.
# Within each window, the count up to time t has a mean that grows with t.
starts = np.arange(W) * h
for t in (1.0, 2.0, 3.0):
    # 각 창의 시작부터 t 시간 안에 들어온 사건 수 / events within t hours of each window's start
    m = np.mean(np.searchsorted(long_path, starts + t) - np.searchsorted(long_path, starts))
    print("E[N(%.0f)] 표본 / sample %.4f   이론 lam*t / theory %.1f" % (t, m, LAM * t))
# 기대값 / Expected  2.0055, 4.0125, 6.0200
#   평균이 t 에 비례해 커집니다. 그러므로 N(t) 는 정상과정이 아닙니다.
#   The mean grows in proportion to t, so N(t) is not a stationary process.
#   변하지 않는 것은 「같은 길이 창의 증분」의 분포입니다. 두 문장을 섞지 마세요.
#   What does not change is the distribution of the increment over a window of a given length.

---
## 7. 합치기와 솎기 / Superposition and thinning

덱 *Slide 28 — Superposition*, *Slide 29 — Thinning*, *Slide 30 — Concept Check. Can We Recover the Original Calls?*
입니다. 여기서도 `arrival_times` 를 다시 짜지 않고 그대로 부릅니다.
*Slides 28 to 30. We call `arrival_times` again rather than rewriting it.*

- **합치기** — 독립인 두 스트림을 합치면 다시 포아송이고 비율이 더해진다. `λ = λ₁ + λ₂`
- **솎기** — 합쳐진 각 사건에 **새 동전**을 던져 확률 `p` 로 남기면 남은 것과 버린 것이 각각 `λp`,
  `λ(1−p)` 의 포아송이다. 되찾는 것은 **비율과 분포**이고, 어느 사건이 원래 어느 회선에서 왔는지가 아니다

**`rng.random(k)`** — 0 과 1 사이 균등 난수를 `k` 개 뽑습니다. `< p` 와 비교하면 확률 `p` 의 동전이 됩니다.
*Draws `k` uniform numbers between 0 and 1. Comparing with `< p` turns them into coins of probability `p`.*
**`np.sort(v)`** — 오름차순으로 정렬합니다. 두 스트림을 합친 뒤 시간순으로 놓을 때 씁니다.
*Sorts ascending. We use it to put the merged stream back in time order.*

In [ ]:
L1, L2, TT = 3.0, 5.0, 100.0        # 두 회선의 비율과 관측 시간 / two lines and the observation span

g = np.random.default_rng(SEED)
line1 = arrival_times(L1, TT, g)     # 회선 1, 시간당 3건 / line 1 at three per hour
line2 = arrival_times(L2, TT, g)     # 회선 2, 시간당 5건 / line 2 at five per hour
merged = np.sort(np.concatenate([line1, line2]))       # 합친 스트림 / the merged stream

print("회선 1 / line 1 :", len(line1), " 이론 L1*TT / theory :", int(L1 * TT))
print("회선 2 / line 2 :", len(line2), " 이론 L2*TT / theory :", int(L2 * TT))
print("합친 것 / merged:", len(merged), " 이론 (L1+L2)*TT     :", int((L1 + L2) * TT))
print("30분 창의 평균 개수 / mean count over a 30-minute window :",
      round(np.bincount(np.floor(merged / 0.5).astype(int), minlength=200).mean(), 4),
      " 이론 8*0.5 / theory : 4")
print()

p = L1 / (L1 + L2)                   # 3/8. 남길 확률 / the probability of keeping an event
keep = g.random(len(merged)) < p     # 사건마다 독립인 동전 / an independent coin per event
print("남긴 것 / kept     :", int(keep.sum()), " 이론 8*(3/8)*TT / theory :", int((L1 + L2) * p * TT))
print("버린 것 / rejected :", int((~keep).sum()), " 이론 8*(5/8)*TT / theory :", int((L1 + L2) * (1 - p) * TT))
# 기대값 / Expected  line1 321, line2 467, merged 788, 30분 평균 3.94, kept 285, rejected 503
#   합쳤다가 3/8 로 솎으면 비율 3 과 5 가 돌아옵니다. 돌아오는 것은 비율이고 출처 표시가 아닙니다.
#   Merging and then thinning with 3/8 returns the rates 3 and 5. The rates come back, the labels do not.

**솎기는 비율을 되찾고 원래 신원은 되찾지 않습니다.** 새로 던진 동전으로 만든 두 스트림은 비율이 맞고 서로 독립이지만,
어느 사건이 원래 어느 회선에서 왔는지는 복원하지 않습니다.
덱 *Slide 30 — Concept Check* 의 마지막 줄입니다. 되찾는 것은 비율과 분포이고 출처 표시가 아닙니다.
*The arithmetic closes but the identities do not come back. Thinning with fresh coins reproduces the rates and gives
two independent outputs. It does not recover which line each real call came from.*

> 솎기의 놀라운 부분은 **남긴 것과 버린 것이 독립**이라는 점입니다. 창 안의 전체 개수를 고정해 놓고 보면
> 둘은 합이 정해져 있으니 당연히 종속입니다.
>
> 다만 **총합이 확률변수이기만 하면 언제나 독립이 되는 것은 아닙니다.** 이 독립은
> 입력이 포아송 과정이고 각 사건에 **독립인 라벨**을 붙였을 때 나오는 thinning 의 결과입니다.
> 덱 *Slide 29 — Thinning* 의 오른쪽 칸이 그 말입니다.
> *The independence here is not a consequence of the total being random on its own. It follows from a Poisson
> input together with one independent label per event. That pair is what thinning assumes.*

---
## 8. Optional / For the curious

여기부터는 이번 주의 핵심이 아니고 과제 1 에도 안 나옵니다. 시간이 남을 때 보세요.
*Not core to this week and not needed for Assignment 1. Look at it if you have time.*

In [ ]:
#@title ▶ Optional A. 비율이 변하면 / When the rate changes { display-mode: "form" }
# 비동차 포아송. 앞 3시간은 시간당 1건, 뒤 3시간은 4건인 6시간 주기.
# A nonhomogeneous Poisson process. One per hour for three hours, then four per hour for three.
def lam_of(t):
    return np.where(t % 6.0 < 3.0, 1.0, 4.0)

LMAX, CYCLES = 4.0, 2000
g = np.random.default_rng(SEED)
candidate = arrival_times(LMAX, 6.0 * CYCLES, g)              # 가장 높은 비율로 후보를 만들고 / propose at the top rate
kept = candidate[g.random(len(candidate)) < lam_of(candidate) / LMAX]   # 비율에 비례해 남긴다 / keep in proportion

in_first = (kept % 6.0) < 3.0
c1 = np.bincount(np.floor(kept[in_first] / 6.0).astype(int), minlength=CYCLES)
c2 = np.bincount(np.floor(kept[~in_first] / 6.0).astype(int), minlength=CYCLES)

print("앞 3시간의 평균 개수 / mean count, first three hours :", round(c1.mean(), 4), " 이론 3 / theory")
print("뒤 3시간의 평균 개수 / mean count, last three hours  :", round(c2.mean(), 4), " 이론 12 / theory")
print("6시간 전체          / the whole six hours           :", round((c1 + c2).mean(), 4), " 이론 15 / theory")
print()
print("조용한 구간에서 잰 비율 1 을 6시간에 곱하면 / measuring the rate 1 in the quiet half and multiplying by six :", 1.0 * 6)
print("비율을 적분하면 3*1 + 3*4 = / integrating the rate gives                                                  :", 3 * 1 + 3 * 4)
# 기대값 / Expected  앞 2.9575, 뒤 12.0015, 전체 14.959
#   같은 길이의 창인데 분포가 다릅니다. 정상증분이 깨진 것입니다. 독립증분은 그대로 남아 있습니다.
#   Two windows of the same length with different distributions. Stationary increments is what broke, not independence.
#   총합만 보면 구간 평균 비율 2.5 를 곱해도 15 가 나옵니다. 총합이 맞는 것과 창 하나의 분포가 맞는 것은 다릅니다.
#   The total also comes out right from the average rate 2.5, but a correct total is not a correct window distribution.

In [ ]:
#@title ▶ Optional B. 사건이 크기를 가지면 / When events carry sizes { display-mode: "form" }
# 복합 포아송. 보험금 청구가 도착하고 건마다 금액이 있다. S(t) = Y_1 + ... + Y_N(t)
# A compound Poisson process. Claims arrive and each carries an amount.
g = np.random.default_rng(SEED)
EY = 1000.0                       # 건당 평균 금액 / mean size per event
EY2 = 2 * EY ** 2                 # 지수분포이면 E[Y^2] = 2 E[Y]^2 / for an exponential size
R = 20_000

N = g.poisson(LAM * T, size=R)                                    # 3시간 동안의 건수 / the count over three hours
S = np.array([g.exponential(scale=EY, size=k).sum() for k in N])   # 그 건들의 금액 합 / the sum of those sizes

print("E[S] 표본 / sample :", round(S.mean(), 1), "   이론 lam*t*E[Y] / theory :", LAM * T * EY)
print("Var[S] 표본 / sample :", round(S.var(ddof=1)), " 이론 lam*t*E[Y^2] / theory :", LAM * T * EY2)
print("틀린 식 lam*t*Var[Y] / the wrong formula :", LAM * T * EY ** 2)
# 기대값 / Expected  E[S] 5980.7, Var[S] 11826631, 이론 12000000, 틀린 식 6000000
#   분산은 lam*t 곱하기 Y 의 이차적률입니다. Y 의 분산이 아닙니다. 여기서는 두 배 차이가 납니다.
#   The variance is lam*t times the second moment of Y, not its variance. Here the two differ by a factor of two.
#   N 으로 조건부를 걸고 분산분해를 적용하면 그 여분의 항이 나옵니다.
#   덱 Slide 41 — Appendix C. Compound Poisson, Events Carry Sizes.
#   Conditioning on N and applying the variance decomposition produces that extra term.
#   Slide 41 — Appendix C. Compound Poisson, Events Carry Sizes.

---
## 9. 직접 해 볼 자리 / Scratch space

과제 1 은 손으로 푸는 과제라 여기의 코드를 제출하지 않습니다. 그래도 과제가 말로 묻는 것을 숫자로 한 번
만져 보면 답을 쓸 때 훨씬 덜 헷갈립니다. 비율을 바꿔 보고, 개수를 세고, 평균과 분산을 비교하고,
간격의 히스토그램을 그려 보세요. 위의 `arrival_times` 를 그대로 부르면 됩니다.
*Assignment 1 is done on paper, so nothing here gets submitted. Even so, handling the same objects in numbers
makes the written answers much less slippery. Change the rate, count, compare the mean with the variance, draw
the gap histogram. Just call `arrival_times` from above.*

In [ ]:
# 9절 연습 공간 / Scratch space for section 9.

---
### 이번 주에 가져갈 것 / What to take away

1. **포아송 모형 아래에서** 비율 하나가 지수 간격과 포아송 개수를 함께 묶습니다.
   비율만으로 분포가 정해지는 것이 아니라, 모형을 하나 고른 뒤에 정해집니다.
   *Under the Poisson model, one rate links exponential gaps and Poisson counts. A rate alone does not
   fix a distribution; a model does.*
2. 그 둘은 같은 타임라인의 두 가지 읽기이고, `P(T₁ > t) = P(N(t) = 0) = e^(−λt)` 가 그 다리입니다.
   *They are two readings of one timeline, and that identity is the bridge.*
3. 이미 기다린 시간은 남은 대기를 바꾸지 않습니다. 이것은 모형의 성질이지 줄 서기의 법칙이 아닙니다.
   *Elapsed time does not change the remaining wait. That is a property of the model, not of real queues.*
4. 평균과 분산이 같다는 것, 상관이 0 이라는 것은 둘 다 증거이지 증명이 아닙니다.
   *Mean equal to variance, and zero correlation, are both evidence rather than proof.*

마지막 셀은 버전을 적어 두는 칸입니다. 숫자가 위의 기대값과 다를 때 이 출력을 같이 보내 주면 원인을
찾기 쉽습니다. 제출하는 것은 없습니다.
*The last cell just records versions. If your numbers differ from the expected values above, send its output
along and the cause is easy to find. Nothing here is submitted.*

In [ ]:
# 버전 기록용입니다. 제출용이 아닙니다 / Records versions. Not a submission cell.
import sys, platform, datetime
print("date    :", datetime.datetime.now().isoformat(timespec="seconds"))
print("python  :", sys.version.split()[0], "on", platform.platform())
print("numpy   :", np.__version__)
import matplotlib
print("mpl     :", matplotlib.__version__)
print("SEED    :", SEED)